# Validation du finetuning sur un résidu *exact* — environnement Crazyflie

Port du notebook de référence `examples/residual_dynamics/exact_residual_offset_demo.ipynb`
**sur l'environnement d'entraînement de `test_cf`** (quad `crazyflie_quad`, politique torch via le
pont `lotf_jax_bridge`, hyperparamètres de `configs/lotf_config.yaml`).

## Idée
On alourdit le Crazyflie de **27 g → 40 g** (le `mass_cycle` du bouton `m` en vol). Une politique
entraînée sur le modèle **léger** (27 g) laisse un **offset stationnaire en z** quand on la vole sur
le quad **lourd** (poussée insuffisante).

Le simulateur intègre `dv/dt = g + R·[0,0,f_d/m_nom] + a_res`. Le résidu **exact** qui corrige le
modèle nominal pour reproduire le quad lourd est analytique :

$$a_{res} = R\,[0,\,0,\,f_d\,(1/m_{lourd} - 1/m_{nom})]$$

## Pourquoi ce test
Contrairement à `diag_replay_ft_env.py` (qui rejoue un résidu **appris** sur un vrai log), ici le
résidu est la **vérité terrain analytique**. C'est donc un test contrôlé de la *brique BPTT* :

- si le finetune corrige l'offset → la chaîne **env BPTT + bptt.train** est saine, et tout problème
  résiduel vient de l'**apprentissage** du résidu (fit MLP) ou du **mismatch sim-to-sim**, pas du BPTT ;
- s'il ne le corrige pas → le problème est dans l'env d'entraînement / les hyperparamètres BPTT.

## Remise au propre
On réutilise au maximum les fonctions **`lotf/`** déjà validées :
`Quadrotor`, `HoveringStateEnv`, les wrappers, `lotf.envs.rollout`, `lotf.algos.bptt.train`,
`env.plot_trajectories`. Les seuls greffons projet sont le pont torch↔flax (`make_lotf_mlp`,
`torch_sd_to_flax`) — indispensable car nos politiques sont en torch + **tanh** — et les
hyperparamètres lus depuis `configs/lotf_config.yaml` / `cf_params.py` (sources uniques).

## 0. Imports et chemins
Même résolution de chemins que `tests/diag_replay_ft_env.py` : on ajoute `core/`, `RL-real/` et la
racine du dépôt au `sys.path`.

In [ ]:
import sys, os, time
from pathlib import Path
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import jax, jax.numpy as jnp
import optax
import torch
import yaml
import matplotlib.pyplot as plt
from flax.training.train_state import TrainState

# --- chemins (comme tests/diag_replay_ft_env.py) ---
THIS = Path.cwd()
REPO = next(p for p in [THIS, *THIS.parents] if (p / "lotf").is_dir())
TEST_CF = REPO / "test_cf"
for p in (TEST_CF / "core", TEST_CF / "RL-real", REPO):
    sys.path.insert(0, str(p))

import cf_params as P                 # constantes plateforme/env (source unique)
from lotf_config import CFG           # hyperparamètres (configs/lotf_config.yaml)

# fonctions lotf déjà validées
from lotf import LOTF_PATH
from lotf.objects import Quadrotor
from lotf.envs import HoveringStateEnv, rollout
from lotf.envs.wrappers import MinMaxObservationWrapper, LogWrapper, VecEnv
from lotf.algos import bptt

# pont torch<->flax du projet (politique torch + non-linéarité tanh)
from lotf_jax_bridge import make_lotf_mlp, torch_sd_to_flax

%matplotlib inline

## 1. Configuration
Masses et cible viennent de `cf_params.py` / `configs/lotf_config.yaml` (pas de magie locale).
La masse lourde est la 2ᵉ valeur du `mass_cycle_kg` (bouton `m` en vol Genesis).

In [ ]:
seed = 0
key = jax.random.key(seed)

M_NOM  = P.MASS                                       # 0.027 kg — Crazyflie nominal
M_REAL = CFG["genesis_bridge"]["mass_cycle_kg"][1]    # 0.040 kg — masse 'lourde' du cycle
hover_target = list(P.HOVER_GOAL)                     # [0, 0, 0.5]

# dynamique 'vraie' du quad lourd (sans résidu)
cfg_real = {"use_high_fidelity": False, "use_forward_residual": False}

# modèle nominal + résidu EXACT reproduisant le quad lourd (analytique : ignore les params)
cfg_nom_res = {
    "use_high_fidelity": False,
    "use_forward_residual": True,
    "exact_mass_residual": {"m_nominal": M_NOM, "m_real": M_REAL},
}
dummy_residual_params = {}   # résidu analytique -> pas de params à passer

print(f"masse nominale = {M_NOM*1000:.0f} g  ->  hover {9.81*M_NOM:.3f} N")
print(f"masse lourde   = {M_REAL*1000:.0f} g  ->  hover {9.81*M_REAL:.3f} N")
print(f"la politique nominale 'croit' qu'il faut {9.81*M_NOM:.3f} N pour planer -> sous-poussée")

## 2. Environnements
- `eval_env_real` : Crazyflie **alourdi à 40 g** (dynamique vraie), pour évaluer les politiques.
  On part du YAML `crazyflie_quad` et on ne surcharge que la masse (`Quadrotor.from_dict`).
- `train_env` (plus bas) : Crazyflie nominal + résidu exact (≡ quad lourd), pour le finetune BPTT.

In [ ]:
EVAL_SIM_TIME = 8.0   # s

with open(LOTF_PATH + "/objects/quadrotor_files/crazyflie_quad.yaml") as f:
    cf_cfg = yaml.safe_load(f)
cf_cfg_heavy = dict(cf_cfg)
cf_cfg_heavy["mass"] = M_REAL                         # Crazyflie alourdi
quad_real = Quadrotor.from_dict(cf_cfg_heavy, cfg_real)

e = CFG["hovering_env"]

def make_eval_env(quad_obj):
    env = HoveringStateEnv(
        max_steps_in_episode=int(EVAL_SIM_TIME / P.DT),
        dt=P.DT, delay=P.DELAY, quad_obj=quad_obj,
        margin=e["margin"], hover_target=hover_target,
    )
    return MinMaxObservationWrapper(env)

eval_env_real = make_eval_env(quad_real)
obs_dim = eval_env_real.observation_space.shape[0]
action_dim = eval_env_real.action_space.shape[0]
print(f"obs_dim={obs_dim}  action_dim={action_dim}")

## 3. Politique de base (pré-entraînée sur le modèle léger)
On charge la politique **torch** `models/model_pretrain.pt` et on la convertit en flax avec le pont
du projet. Le biais hover du MLP est le biais **nominal** (`P.HOVERING_ACTION`), celui avec lequel la
politique torch a été pré-entraînée — c'est cohérent avec `finetune_lotf_jax.py` / `flax_to_torch_sd`.

In [ ]:
BASE_PT = TEST_CF / "RL-real" / "model_pretrain.pt"
ckpt = torch.load(BASE_PT, map_location="cpu")
sd = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
base_policy_params = torch_sd_to_flax(sd)

policy_net = make_lotf_mlp(obs_dim, action_dim, P.HOVERING_ACTION)   # biais hover NOMINAL + tanh

def make_policy_fn(params):
    def policy(obs, key):
        return policy_net.apply(params, obs)
    return policy

policy_nominal = make_policy_fn(base_policy_params)
print(f"politique de base : {BASE_PT.name}")

## 4. Helpers d'évaluation
`rollout` est la fonction lotf ; on la vmappe sur plusieurs graines (comme le notebook de référence).

> **Masquage post-`done`.** `lotf.envs.rollout` tourne sans auto-reset : une fois qu'une trajectoire
> sort de la boîte de sécurité (`terminated`/`truncated`), le simulateur **continue d'intégrer** et le
> quad part en chute libre (z → ±10 m). Ces valeurs post-crash n'ont pas de sens physique. On les
> **masque** (NaN après le 1ᵉʳ `done`) — exactement ce que fait `env.plot_trajectories` en coupant
> chaque trajectoire à son premier `done`. Sans ce masque, l'enveloppe min–max et l'offset moyen sont
> contaminés par les divergences.

In [ ]:
def get_rollouts(env, policy, num_rollouts=20, key=jax.random.key(0)):
    keys = jax.random.split(key, num_rollouts)
    parallel_rollout = jax.vmap(rollout, in_axes=(None, 0, None, None))
    return parallel_rollout(env, keys, policy, dummy_residual_params)

def z_valid(transitions):
    """Altitude z masquée APRÈS le 1ᵉʳ done (NaN), + temps. (num_rollouts, T).
    Même coupe que env.plot_trajectories -> écarte les divergences post-crash."""
    z = np.asarray(transitions.state.quadrotor_state.p[:, :, 2])
    t = np.asarray(transitions.state.time[0])
    done = np.asarray(transitions.terminated) | np.asarray(transitions.truncated)
    T = z.shape[1]
    # 1ᵉʳ done par rollout (T-1 si jamais done) ; on garde t <= 1ᵉʳ done
    first_done = np.where(done.any(1), done.argmax(1), T - 1)
    keep = np.arange(T)[None, :] <= first_done[:, None]
    return t, np.where(keep, z, np.nan)

def report_offset(transitions, label):
    _, z = z_valid(transitions)
    # régime établi = moyenne des 50 derniers pas VALIDES de chaque rollout
    finals = []
    for zi in z:
        v = zi[~np.isnan(zi)]
        if v.size:
            finals.append(v[-50:].mean())
    final_z = float(np.mean(finals))
    offset = final_z - hover_target[2]
    print(f"{label:38s} z_final \u2248 {final_z:.3f} m   (cible {hover_target[2]}, offset {offset:+.3f} m)")
    return final_z

## 5. Baseline — politique nominale sur le Crazyflie lourd → offset en z

Le graphe par trajectoire (`plot_trajectories`) coupe déjà chaque vol à son premier `done` : on y voit
que plusieurs rollouts **terminent tôt** (la politique non adaptée gère mal le quad lourd).

In [ ]:
tr_baseline = get_rollouts(eval_env_real, policy_nominal)
report_offset(tr_baseline, "base @ Crazyflie 40 g (baseline)")
eval_env_real.plot_trajectories(tr_baseline)

## 6. Finetune sur (modèle nominal + résidu exact) ≡ quad lourd
On part de la politique de base et on la ré-entraîne par BPTT (`lotf.algos.bptt.train`) sur la
dynamique corrigée par le résidu exact. Même env d'entraînement que `finetune_lotf_jax.py`
(`HoveringStateEnv(crazyflie_quad, use_forward_residual)` + randomisation `hovering_env`), avec
schedule cosine + grad-clip de `configs/lotf_config.yaml`. `res_model_params` est ignoré (résidu analytique).

> Note : on fait ici un **tir unique long** (`max_epochs=400`) pour valider la correction complète.
> En vol, la pipeline est **incrémentale** (`CFG['online']` : 30 epochs/step, plusieurs steps).

In [ ]:
num_envs       = CFG["bptt"]["num_envs"]        # 10 (article)
max_epochs     = 400                            # tir unique long (validation)
train_sim_time = CFG["bptt"]["max_sim_time"]    # 3 s

quad_train = Quadrotor.from_name(CFG["bptt"]["quad"], cfg_nom_res)   # crazyflie nominal + résidu exact
train_env = HoveringStateEnv(
    max_steps_in_episode=int(train_sim_time / P.DT), dt=P.DT, delay=P.DELAY,
    yaw_scale=e["yaw_scale"], pitch_roll_scale=e["pitch_roll_scale"],
    velocity_std=e["velocity_std"], omega_std=e["omega_std"], quad_obj=quad_train,
    reward_sharpness=e["reward_sharpness"], action_penalty_weight=e["action_penalty_weight"],
    margin=e["margin"], hover_target=hover_target,
)
train_env = VecEnv(LogWrapper(MinMaxObservationWrapper(train_env)))

scheduler = optax.cosine_decay_schedule(CFG["bptt"]["lr"], max_epochs)
tx = optax.chain(optax.clip_by_global_norm(CFG["bptt"]["grad_clip"]), optax.adam(scheduler))
train_state = TrainState.create(apply_fn=policy_net.apply, params=base_policy_params, tx=tx)

key, key_bptt = jax.random.split(key)
init_env_state, init_obs = train_env.reset(jax.random.split(key_bptt, num_envs), None)

t0 = time.time()
res_dict = bptt.train(
    train_env, init_env_state, init_obs, train_state,
    num_epochs=max_epochs, num_steps_per_epoch=train_env.max_steps_in_episode,
    num_envs=num_envs, res_model_params=dummy_residual_params, key=key_bptt,
)
print(f"Finetune terminé en {time.time() - t0:.1f} s")

finetuned_params = res_dict["runner_state"].train_state.params
policy_finetuned = make_policy_fn(finetuned_params)

## 7. Éval — politique finetunée sur le Crazyflie lourd → offset corrigé

In [ ]:
tr_finetuned = get_rollouts(eval_env_real, policy_finetuned)
report_offset(tr_finetuned, "finetunée (résidu exact) @ 40 g")
eval_env_real.plot_trajectories(tr_finetuned)

## 8. Comparaison de l'altitude : impact du finetune
Courbe = moyenne sur les rollouts (post-`done` masqué) ; enveloppe = min–max **des trajectoires
valides** (avant divergence). L'enveloppe se rétrécit dans le temps : moins de rollouts encore en vol.

In [ ]:
z0 = report_offset(tr_baseline,  "base @ 40 g")
z1 = report_offset(tr_finetuned, "finetunée @ 40 g")

t, zb = z_valid(tr_baseline)     # NaN après le 1ᵉʳ done
_, zf = z_valid(tr_finetuned)

def band(ax, t, z, color, label):
    ax.plot(t, np.nanmean(z, 0), color=color, lw=2, label=label)
    ax.fill_between(t, np.nanmin(z, 0), np.nanmax(z, 0), color=color, alpha=0.15)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(hover_target[2], color="k", ls="--", lw=1, label=f"cible z = {hover_target[2]}")
band(ax, t, zb, "#e74c3c", f"base @ lourd (z\u2248{z0:.3f})")
band(ax, t, zf, "#27ae60", f"finetunée+résidu @ lourd (z\u2248{z1:.3f})")
ax.set_xlabel("temps (s)"); ax.set_ylabel("altitude z (m)")
ax.set_title("Crazyflie 27 g \u2192 40 g : correction de l'offset z par finetune (résidu exact)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

print(f"\nVERDICT : offset {z0-hover_target[2]:+.3f} m  ->  {z1-hover_target[2]:+.3f} m")

## 9. Conclusion

Si l'offset z passe d'une valeur négative nette (baseline, sous-poussée) à ~0 après finetune, alors
**la brique BPTT de `test_cf` est validée** : sur une vérité-terrain analytique, `bptt.train` contre
`HoveringStateEnv(crazyflie_quad)` + résidu corrige bien la dynamique.

→ Tout offset résiduel observé dans Genesis / sur le vrai drone ne vient donc **pas** de la mécanique
BPTT, mais soit de l'**apprentissage du résidu** (fit MLP sur peu de samples, cf. `A Faire.txt`), soit
d'un **mismatch sim-to-sim** (delay/actionneur), conclusion cohérente avec `diag_replay_ft_env.py`.

On note aussi qu'en **baseline** plusieurs rollouts terminent tôt (le quad lourd non adapté est dur à
tenir) tandis qu'après finetune les vols restent dans la boîte : le finetune améliore la **stabilité**,
pas seulement l'offset moyen.